In [1]:
set.seed(405)
options(stringsAsFactors = FALSE)

In [2]:
required_packages <- c("gssr", "dplyr", "tidyr", "readr", "tibble")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages) > 0) install.packages(missing_packages, repos = "https://cloud.r-project.org")
invisible(lapply(required_packages, library, character.only = TRUE))

Package loaded. To attach the GSS data, type data(gss_all) at the console.
For the panel data and documentation, type e.g. data(gss_panel08_long) and data(gss_panel_doc).
For help on a specific GSS variable, type ?varname at the console.

Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘tibble’ was built under R version 4.4.3”


In [3]:
gss_2024 <- gssr::gss_get_yr(2024)
core_vars <- c("id", "confinan", "polviews", "degree", "age", "sex", "income", "region")
gss_core <- gss_2024 %>% dplyr::select(dplyr::any_of(core_vars))

Fetching: https://gss.norc.org/documents/stata/2024_stata.zip



In [4]:
dplyr::glimpse(gss_core)

Rows: 3,986
Columns: 8
$ id       <dbl> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18…
$ confinan <dbl+lbl>     1,     2,     1, NA(i),     2,     1,     2,     3,  …
$ polviews <dbl+lbl>     4,     3,     1, NA(d),     4, NA(d),     2,     4,  …
$ degree   <dbl+lbl> 3, 4, 2, 1, 1, 2, 1, 1, 2, 1, 3, 0, 2, 1, 1, 4, 3, 1, 0, …
$ age      <dbl+lbl>    33,    64,    69,    19,    70,    53,    48,    30,  …
$ sex      <dbl+lbl>     1,     1,     2,     1,     2,     1,     2,     2,  …
$ income   <dbl+lbl>    12,    12,    12,    12,    12,     7,    12,     8,  …
$ region   <dbl+lbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …


In [5]:
missingness <- gss_core %>%
  dplyr::summarise(dplyr::across(dplyr::everything(), ~sum(is.na(.)))) %>%
  tidyr::pivot_longer(cols = dplyr::everything(), names_to = "variable", values_to = "n_missing") %>%
  dplyr::mutate(pct_missing = n_missing / nrow(gss_core)) %>%
  dplyr::arrange(dplyr::desc(pct_missing))
missingness

variable,n_missing,pct_missing
<chr>,<int>,<dbl>
confinan,1372,0.344204717
income,423,0.106121425
polviews,164,0.041144004
age,127,0.031861515
sex,30,0.007526342
degree,7,0.001756147
id,0,0.000000000
region,0,0.000000000


In [6]:
value_counts <- dplyr::bind_rows(
  gss_core %>% dplyr::count(confinan, name = "n") %>% dplyr::mutate(variable = "confinan", level = as.character(confinan)),
  gss_core %>% dplyr::count(polviews, name = "n") %>% dplyr::mutate(variable = "polviews", level = as.character(polviews)),
  gss_core %>% dplyr::count(degree, name = "n") %>% dplyr::mutate(variable = "degree", level = as.character(degree)),
  gss_core %>% dplyr::count(sex, name = "n") %>% dplyr::mutate(variable = "sex", level = as.character(sex)),
  gss_core %>% dplyr::count(income, name = "n") %>% dplyr::mutate(variable = "income", level = as.character(income)),
  gss_core %>% dplyr::count(region, name = "n") %>% dplyr::mutate(variable = "region", level = as.character(region))
) %>%
  dplyr::select(variable, level, n) %>%
  dplyr::arrange(variable, dplyr::desc(n))
value_counts

variable,level,n
<chr>,<chr>,<int>
confinan,2,1487
confinan,NA,1372
confinan,3,684
confinan,1,443
degree,1,1817
degree,3,863
degree,4,577
degree,0,362
degree,2,360


In [7]:
audit_summary <- tibble::tibble(
  metric = c("n_rows_raw", "n_cols_raw", "n_rows_core", "n_complete_core"),
  value = c(nrow(gss_2024), ncol(gss_2024), nrow(gss_core), sum(stats::complete.cases(gss_core)))
)
audit_summary

metric,value
<chr>,<int>
n_rows_raw,3986
n_cols_raw,980
n_rows_core,3986
n_complete_core,2223


In [8]:
dir.create("../output/audit", recursive = TRUE, showWarnings = FALSE)
readr::write_csv(missingness, "../output/audit/missingness_2024_core.csv")
readr::write_csv(value_counts, "../output/audit/value_counts_2024_core.csv")
readr::write_csv(audit_summary, "../output/audit/audit_summary_2024_core.csv")